In [32]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import ast
import matplotlib.pyplot as plt

In [25]:
HORIZON = 24
SEQ_LEN = 48
EPOCHS = 50
patience = 0
LR = 0.0001

In [26]:
df_main = pd.read_csv("final_data.csv")
df_main["datetime"] = pd.to_datetime(df_main["datetime"])

dk = pd.read_csv("hourly_data_Danmark_55.99775695800781_10.0052490234375.csv")
ring = pd.read_csv("hourly_data_Ringsted_56.1740608215332_9.545852661132812.csv")
silk = pd.read_csv("hourly_data_Silkeborg_55.444580078125_11.78314208984375.csv")

for d in [dk, ring, silk]:
    d["date"] = pd.to_datetime(d["date"])

dk = dk.add_prefix("dk_").rename(columns={"dk_date": "datetime"})
ring = ring.add_prefix("ring_").rename(columns={"ring_date": "datetime"})
silk = silk.add_prefix("silk_").rename(columns={"silk_date": "datetime"})

df = df_main.merge(dk, on="datetime", how="left")
df = df.merge(ring, on="datetime", how="left")
df = df.merge(silk, on="datetime", how="left")

df = df.sort_values("datetime")

df["hour"] = df["datetime"].dt.hour
df["lag_1"] = df["value_spot"].shift(1)
df["lag_2"] = df["value_spot"].shift(2)
df["lag_24"] = df["value_spot"].shift(24)

df = df.ffill()

df = pd.get_dummies(
    df,
    columns=[c for c in ["zone_prev", "zone"] if c in df.columns],
    drop_first=True
)

df["import"] = df["import"].apply(ast.literal_eval)
df["export"] = df["export"].apply(ast.literal_eval)

df = pd.concat([
    df.drop(columns=["import", "export"]),
    df["import"].apply(pd.Series).add_prefix("import_"),
    df["export"].apply(pd.Series).add_prefix("export_")
], axis=1)

df["mix"] = df["mix"].apply(ast.literal_eval)
df = pd.concat([
    df.drop(columns=["mix"]),
    df["mix"].apply(pd.Series)
], axis=1)

df = df.join(df["flows"].apply(pd.Series))

df = df.fillna(0)

df = df.drop(columns=[
    "emissionFactorType",
    "battery storage",
    "hydro storage",
    "flows",
    "unit_spot",
    "unit_total_load"
])
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["weekday_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["weekday_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["month_sin"] = np.sin(2 * np.pi * (df["month"] -1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] -1)/ 12)


for i in range(1, HORIZON + 1):
    df[f"price_t+{i}"] = df["value_spot"].shift(-i)
    df[f"carbon_t+{i}"] = df["carbonIntensity"].shift(-i)

hours = df["hour"].values
df = df.drop(columns=["datetime"], errors="ignore")
df = df.dropna()

price_cols = [f"price_t+{i}" for i in range(1, HORIZON + 1)]
carbon_cols = [f"carbon_t+{i}" for i in range(1, HORIZON + 1)]

X = df.drop(columns=price_cols + carbon_cols).values
Y_price = df[price_cols].values
Y_carbon = df[carbon_cols].values

hours = hours[-len(X):]

split1 = int(0.7 * len(X))
split2 = int(0.85 * len(X))

X_train, X_val, X_test = X[:split1], X[split1:split2], X[split2:]
Yp_train, Yp_val, Yp_test = Y_price[:split1], Y_price[split1:split2], Y_price[split2:]
Yc_train, Yc_val, Yc_test = Y_carbon[:split1], Y_carbon[split1:split2], Y_carbon[split2:]
h_train, h_val, h_test = hours[:split1], hours[split1:split2], hours[split2:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

def create_sequences_12(X, Y_price, Y_carbon, hours, seq_len=48):
    X_seq, Yp_seq, Yc_seq = [], [], []

    for i in range(len(X) - seq_len - HORIZON):
        if hours[i + seq_len - 1] != 12:
            continue

        X_seq.append(X[i:i + seq_len])
        Yp_seq.append(Y_price[i + seq_len])
        Yc_seq.append(Y_carbon[i + seq_len])

    return np.array(X_seq), np.array(Yp_seq), np.array(Yc_seq)


/var/folders/xw/j6c1c3514_l7lyj6wyc6l1rc0000gn/T/ipykernel_36155/3580407337.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


In [27]:
X_train, Yp_train, Yc_train = create_sequences_12(X_train, Yp_train, Yc_train, h_train, SEQ_LEN)
X_val, Yp_val, Yc_val = create_sequences_12(X_val, Yp_val, Yc_val, h_val, SEQ_LEN)
X_test, Yp_test, Yc_test = create_sequences_12(X_test, Yp_test, Yc_test, h_test, SEQ_LEN)

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader_price = DataLoader(SeqDataset(X_train, Yp_train), batch_size=64, shuffle=True)
val_loader_price = DataLoader(SeqDataset(X_val, Yp_val), batch_size=64)

train_loader_carbon = DataLoader(SeqDataset(X_train, Yc_train), batch_size=64, shuffle=True)
val_loader_carbon = DataLoader(SeqDataset(X_val, Yc_val), batch_size=64)

test_loader_price = DataLoader(SeqDataset(X_test, Yp_test), batch_size=64)
test_loader_carbon = DataLoader(SeqDataset(X_test, Yc_test), batch_size=64)

In [28]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        pe = torch.zeros(5000, d_model)
        pos = torch.arange(0, 5000).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        d_model = 256
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)

        self.fc = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, HORIZON)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

model_price = TransformerModel(X_train.shape[2])
model_carbon = TransformerModel(X_train.shape[2])

criterion = nn.L1Loss()

opt_price = torch.optim.Adam(model_price.parameters(), lr=LR)
opt_carbon = torch.optim.Adam(model_carbon.parameters(), lr=LR)

In [29]:
def train(model, optimizer, train_loader, val_loader, patience):
    best_val_loss = float("inf")
    counter = 0
    best_weights = None

    for epoch in range(EPOCHS):

        model.train()
        train_loss = 0.0

        for Xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()


        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for Xb, yb in val_loader:
                val_loss += criterion(model(Xb), yb).item()

        print(f"Epoch {epoch+1}: Train {train_loss:.2f}, Val {val_loss:.2f}")


        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping")
                break


    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model

print("Training PRICE model")
train(model_price, opt_price, train_loader_price, val_loader_price, patience)

print("\nTraining CARBON model")
train(model_carbon, opt_carbon, train_loader_carbon, val_loader_carbon, patience)

Training PRICE model
Epoch 1: Train 2754.71, Val 300.30
Epoch 2: Train 2739.48, Val 298.61
Epoch 3: Train 2735.45, Val 296.54
Epoch 4: Train 2726.41, Val 293.82
Epoch 5: Train 2709.88, Val 290.40
Epoch 6: Train 2694.28, Val 286.18
Epoch 7: Train 2671.03, Val 281.16
Epoch 8: Train 2646.99, Val 275.33
Epoch 9: Train 2621.30, Val 268.70
Epoch 10: Train 2589.51, Val 261.33
Epoch 11: Train 2554.40, Val 253.30
Epoch 12: Train 2512.31, Val 244.61
Epoch 13: Train 2473.34, Val 235.52
Epoch 14: Train 2429.09, Val 225.96
Epoch 15: Train 2378.69, Val 215.95
Epoch 16: Train 2331.09, Val 205.71
Epoch 17: Train 2282.03, Val 195.26
Epoch 18: Train 2227.35, Val 184.77
Epoch 19: Train 2175.08, Val 174.35
Epoch 20: Train 2121.48, Val 164.19
Epoch 21: Train 2066.12, Val 154.25
Epoch 22: Train 2009.85, Val 145.25
Epoch 23: Train 1955.81, Val 137.42
Epoch 24: Train 1900.51, Val 131.31
Epoch 25: Train 1851.39, Val 127.13
Epoch 26: Train 1805.61, Val 125.42
Epoch 27: Train 1758.54, Val 126.57
Early stopping



TransformerModel(
  (input_proj): Linear(in_features=78, out_features=256, bias=True)
  (pos): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=24, bias=True)
  )
)

In [33]:
def evaluate(model, loader):
    preds, actuals = [], []
    model.eval()

    with torch.no_grad():
        for Xb, yb in loader:
            preds.append(model(Xb).numpy())
            actuals.append(yb.numpy())

    return np.vstack(preds), np.vstack(actuals)

price_pred, price_true = evaluate(model_price, test_loader_price)
carbon_pred, carbon_true = evaluate(model_carbon, test_loader_carbon)

print("\nFINAL RESULTS")
print(f"Price MAE: {np.mean(np.abs(price_pred - price_true)):.2f}")
print(f"Carbon MAE: {np.mean(np.abs(carbon_pred - carbon_true)):.2f}")


FINAL RESULTS
Price MAE: 44.34
Carbon MAE: 68.88
